# 01 · Data preparation

Reads the raw sources in `data/raw/` with Spark, cleans them and writes analysis-ready Parquet tables
to `data/processed/`. Run `python scripts/download_data.py` first.

| Source | Raw input | Output table |
|---|---|---|
| Dengue surveillance (CDC Perú) | `dengue/dengue_surveillance_2000_2025.csv` | `dengue_cases` (partitioned by `year`) |
| Population with DNI (RENIEC) | `reniec/population_dni_*.csv` | `population` |
| Water emergencies (SUNASS) | `sunass/water_emergencies.csv` | `water_emergencies` |
| Health facilities (RENIPRESS) | `renipress/renipress_2026-09-30.csv` | `health_facilities` |
| Weather stations (SENAMHI) | `senamhi/*/station.json`, `senamhi/*/YYYYMM.json` | `weather_daily` |

Every source is read with an **explicit schema** instead of `inferSchema`: it avoids an extra pass over
the file, guarantees the types, and keeps codes such as `ubigeo` as text so leading zeros are not lost.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "catalog.yaml").exists())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

spark = (SparkSession.builder
         .master("local[*]")
         .appName("dengue-data-preparation")
         .config("spark.sql.session.timeZone", "America/Lima")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 20:58:23 WARN Utils: Your hostname, Raymonds-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.18.58 instead (on interface en0)
26/10/30 20:58:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/30 20:58:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Shared helpers

- `write_table` writes a DataFrame as Parquet and reads it back, so every check below runs on what was
  actually written.
- Peru's epidemiological weeks run Sunday to Saturday; week 1 is the week that contains January 4.
  `epi_week_start` turns (year, week) into the week's Sunday; `epi_year_week` does the inverse for a date.
- `life_stage` groups ages into MINSA's life stages, used for both cases and population.

In [2]:
def write_table(df, name, partition_by=None):
    # Write df to data/processed/<name>, replacing previous output, and return the written table.
    path = str(PROCESSED / name)
    if partition_by:
        # One file per partition folder; this repartition is a shuffle.
        df.repartition(partition_by).write.mode("overwrite").partitionBy(partition_by).parquet(path)
    else:
        df.coalesce(1).write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


def epi_week_start(year, week):
    jan4 = F.make_date(year, F.lit(1), F.lit(4))
    week1_start = F.date_sub(jan4, F.dayofweek(jan4) - 1)  # dayofweek: 1 = Sunday
    return F.date_add(week1_start, (week - 1) * 7)


def epi_year_week(date):
    # A week belongs to the year of its Wednesday.
    wednesday = F.date_add(date, 4 - F.dayofweek(date))
    return F.year(wednesday), (F.floor((F.dayofyear(wednesday) - 1) / 7) + 1).cast("int")


def life_stage(age):
    # MINSA life stages: niño 0-11, adolescente 12-17, joven 18-29, adulto 30-59, adulto mayor 60+.
    return (F.when(age < 12, "00-11").when(age < 18, "12-17").when(age < 30, "18-29")
             .when(age < 60, "30-59").when(age >= 60, "60+"))

In [3]:
# Known first days of epidemiological week 1 (Sunday-start weeks containing January 4).
known = spark.createDataFrame([(2022, "2022-01-02"), (2023, "2023-01-01"), (2024, "2023-12-31"), (2025, "2024-12-29")],
                              "year INT, expected STRING")
check = known.select("year", F.to_date("expected").alias("expected"),
                     epi_week_start(F.col("year"), F.lit(1)).alias("computed"))
check.show()
assert check.filter(F.col("expected") != F.col("computed")).count() == 0


[Stage 0:>                                                          (0 + 1) / 1]



+----+----------+----------+
|year|  expected|  computed|
+----+----------+----------+
|2022|2022-01-02|2022-01-02|
|2023|2023-01-01|2023-01-01|
|2024|2023-12-31|2023-12-31|
|2025|2024-12-29|2024-12-29|
+----+----------+----------+



## 1. Dengue cases

One row per reported case (2000–2025). Column names are translated to English; values stay as published.

- `age_years`: age normalized to years (`tipo_edad` says whether `edad` is in years, months or days).
- `severity`: from the ICD-10 code (A97.0 without warning signs, A97.1 with warning signs, A97.2 severe).
- `week_start`: Sunday of the epidemiological week, to line up cases with weather data.

These are transformations only: Spark reads nothing until an action (`show`, `count`, `write`) runs.

In [4]:
dengue_raw = spark.read.csv(
    str(RAW / "dengue" / "dengue_surveillance_2000_2025.csv"),
    header=True,
    schema="""departamento STRING, provincia STRING, distrito STRING, localidad STRING, enfermedad STRING,
              ano INT, semana INT, diagnostic STRING, tipo_dx STRING, diresa STRING, ubigeo STRING,
              localcod STRING, edad INT, tipo_edad STRING, sexo STRING""")
dengue_raw.show(3, truncate=False)

+------------+-------------+----------------------+-------------------------+---------------------------+----+------+----------+-------+------+------+--------+----+---------+----+
|departamento|provincia    |distrito              |localidad                |enfermedad                 |ano |semana|diagnostic|tipo_dx|diresa|ubigeo|localcod|edad|tipo_edad|sexo|
+------------+-------------+----------------------+-------------------------+---------------------------+----+------+----------+-------+------+------+--------+----+---------+----+
|HUANUCO     |LEONCIO PRADO|LUYANDO               |SANTA ROSA DE SHAPAJILLLA|DENGUE SIN SIGNOS DE ALARMA|2000|47    |A97.0     |P      |10    |100605|NULL    |9   |A        |M   |
|HUANUCO     |LEONCIO PRADO|LUYANDO               |NARANJILLO               |DENGUE SIN SIGNOS DE ALARMA|2000|40    |A97.0     |P      |10    |100605|NULL    |18  |A        |F   |
|HUANUCO     |LEONCIO PRADO|JOSE CRESPO Y CASTILLO|AUCAYACU                 |DENGUE SIN SIGNOS DE AL

In [5]:
age_years = (F.when(F.col("tipo_edad") == "A", F.col("edad"))
              .when(F.col("tipo_edad") == "M", F.col("edad") / 12)
              .when(F.col("tipo_edad") == "D", F.col("edad") / 365.25))

dengue = (dengue_raw
          .select(F.col("ano").alias("year"),
                  F.col("semana").alias("epi_week"),
                  "ubigeo",
                  F.col("departamento").alias("department"),
                  F.col("provincia").alias("province"),
                  F.col("distrito").alias("district"),
                  F.col("diagnostic").alias("icd10"),
                  F.col("tipo_dx").alias("diagnosis_type"),
                  F.col("sexo").alias("sex"),
                  age_years.cast("double").alias("age_years"))
          .withColumn("severity", F.when(F.col("icd10") == "A97.0", "without_warning_signs")
                                   .when(F.col("icd10") == "A97.1", "with_warning_signs")
                                   .when(F.col("icd10") == "A97.2", "severe"))
          .withColumn("is_confirmed", F.col("diagnosis_type") == "C")
          .withColumn("age_group", life_stage(F.col("age_years")))
          .withColumn("week_start", epi_week_start(F.col("year"), F.col("epi_week"))))

dengue_cases = write_table(dengue, "dengue_cases", partition_by="year")
print(f"{dengue_cases.count():,} cases written")
dengue_cases.groupBy("severity").count().orderBy(F.desc("count")).show()
dengue_cases.groupBy("age_group").count().orderBy("age_group").show()


[Stage 9:>                                                          (0 + 9) / 9]



1,065,274 cases written


+--------------------+------+
|            severity| count|
+--------------------+------+
|without_warning_s...|944670|
|  with_warning_signs|116407|
|              severe|  4197|
+--------------------+------+

+---------+------+
|age_group| count|
+---------+------+
|    00-11|176765|
|    12-17|146794|
|    18-29|247616|
|    30-59|401017|
|      60+| 93082|
+---------+------+



## 2. Population (RENIEC)

One file per December snapshot (2022–2025). The snapshot year comes from the file name through Spark's
hidden `_metadata.file_path` column. Peruvians living abroad are dropped, and people are summed by
district, sex and life stage (the two DNI types are merged).

In [6]:
reniec_raw = spark.read.csv(
    str(RAW / "reniec" / "population_dni_*.csv"),
    header=True,
    schema="""Residencia STRING, Cod_continente STRING, Continente STRING, CodPais STRING, Pais STRING,
              UBIGEO_RENIEC STRING, UBIGEO_INEI STRING, Departamento STRING, Provincia STRING,
              Distrito STRING, TipoDNI STRING, Sexo STRING, Edad INT, Cantidad LONG""")

population = (reniec_raw
              .select("*", F.col("_metadata.file_path").alias("source_file"))
              .filter(F.col("Residencia") == "Nacional")
              .select(F.regexp_extract("source_file", r"population_dni_(\d{4})-12", 1).cast("int").alias("year"),
                      F.col("UBIGEO_INEI").alias("ubigeo"),
                      F.upper("Departamento").alias("department"),
                      F.upper("Provincia").alias("province"),
                      F.upper("Distrito").alias("district"),
                      F.when(F.col("Sexo") == "Hombre", "M").when(F.col("Sexo") == "Mujer", "F").alias("sex"),
                      life_stage(F.col("Edad")).alias("age_group"),
                      F.col("Cantidad").alias("people"))
              .groupBy("year", "ubigeo", "department", "province", "district", "sex", "age_group")
              .agg(F.sum("people").alias("population")))

population = write_table(population, "population")
print(f"{population.count():,} rows written")
population.groupBy("year").agg(F.countDistinct("ubigeo").alias("districts"),
                               F.sum("population").alias("population")).orderBy("year").show()

75,636 rows written
+----+---------+----------+
|year|districts|population|
+----+---------+----------+
|2022|     1890|  35063104|
|2023|     1891|  35411932|
|2024|     1891|  35652954|
|2025|     1892|  35843232|
+----+---------+----------+



## 3. Water emergencies (SUNASS)

One row per district included in an emergency decree. The file is Latin-1 encoded. Dates come as
`yyyyMMdd`. `emergency_type` is derived from the short description: intense rain, water deficit
(drought) or other (cold spells); `is_extension` flags decree extensions (*prórrogas*), which repeat
districts and must be aggregated before joining with cases.

In [7]:
sunass_raw = spark.read.csv(
    str(RAW / "sunass" / "water_emergencies.csv"),
    header=True,
    encoding="ISO-8859-1",
    schema="""ID INT, DEPARTAMENTO STRING, PROVINCIA STRING, DISTRITO STRING, UBIGEO STRING, POBLACION INT,
              COB_AGUA DOUBLE, CONT_HD DOUBLE, DS_EMERG STRING, ANIO INT, CORREL STRING, FECHA_INICIO STRING,
              PLAZO_DIAS INT, FECHA_FINAL STRING, DESC_BREVE STRING, DESC_DET STRING""")

water_emergencies = (sunass_raw
                     .select(F.col("UBIGEO").alias("ubigeo"),
                             F.col("DEPARTAMENTO").alias("department"),
                             F.col("PROVINCIA").alias("province"),
                             F.col("DISTRITO").alias("district"),
                             F.col("ANIO").alias("year"),
                             F.col("DS_EMERG").alias("decree"),
                             F.to_date("FECHA_INICIO", "yyyyMMdd").alias("start_date"),
                             F.to_date("FECHA_FINAL", "yyyyMMdd").alias("end_date"),
                             F.col("PLAZO_DIAS").alias("duration_days"),
                             F.col("POBLACION").alias("district_population"),
                             F.col("COB_AGUA").alias("water_coverage_pct"),
                             F.col("CONT_HD").alias("water_hours_per_day"),
                             F.col("DESC_BREVE").alias("description"))
                     .withColumn("emergency_type", F.when(F.col("description").rlike("(?i)d[ée]ficit"), "drought")
                                                    .when(F.col("description").rlike("(?i)precipitaci|lluvia"), "rain")
                                                    .otherwise("other"))
                     .withColumn("is_extension", F.col("description").rlike("(?i)^pr[óo]rroga")))

water_emergencies = write_table(water_emergencies, "water_emergencies")
print(f"{water_emergencies.count():,} rows written")
water_emergencies.groupBy("year", "emergency_type").agg(F.countDistinct("ubigeo").alias("districts"),
                                                        F.count("*").alias("rows")).orderBy("year", "emergency_type").show()

9,538 rows written
+----+--------------+---------+----+
|year|emergency_type|districts|rows|
+----+--------------+---------+----+
|2017|          rain|      664| 868|
|2023|       drought|     1005|1894|
|2023|         other|      280| 280|
|2023|          rain|     1268|5136|
|2024|          rain|      870|1360|
+----+--------------+---------+----+



## 4. Health facilities (RENIPRESS)

Semicolon-separated, one row per facility. Some addresses contain line breaks and quotes written as
`""` (the CSV standard), while Spark expects `\\"` by default; without `escape='"'` and `multiLine=True`
two rows are split and three active facilities are lost. Only active facilities are kept. Category `0` means
"no category" and becomes null; `care_level` keeps the level (I, II or III) without the sub-level.

In [8]:
renipress_raw = spark.read.csv(
    str(RAW / "renipress" / "renipress_2026-09-30.csv"),
    header=True,
    sep=";",
    escape='"',
    multiLine=True,
    schema="""INSTITUCION STRING, COD_IPRESS STRING, NOMBRE STRING, CLASIFICACION STRING,
              TIPO_ESTABLECIMIENTO STRING, DEPARTAMENTO STRING, PROVINCIA STRING, DISTRITO STRING,
              UBIGEO STRING, DIRECCION STRING, CO_DISA STRING, COD_RED STRING, COD_MICRORRED STRING,
              DISA STRING, RED STRING, MICRORED STRING, COD_UE STRING, UNIDAD_EJECUTORA STRING,
              CATEGORIA STRING, TELEFONO STRING, HORARIO STRING, INICIO_ACTIVIDAD STRING, ESTADO STRING,
              NORTE DOUBLE, ESTE DOUBLE, IMAGEN_1 STRING, FE_ACT_IMAGEN_1 STRING, IMAGEN_2 STRING,
              FE_ACT_IMAGEN_2 STRING, IMAGEN_3 STRING, FE_ACT_IMAGEN_3 STRING""")

health_facilities = (renipress_raw
                     .filter(F.col("ESTADO") == "ACTIVO")
                     .select(F.col("COD_IPRESS").alias("facility_code"),
                             F.col("NOMBRE").alias("name"),
                             F.col("INSTITUCION").alias("institution"),
                             F.col("CLASIFICACION").alias("classification"),
                             F.col("TIPO_ESTABLECIMIENTO").alias("facility_type"),
                             F.when(F.col("CATEGORIA") != "0", F.col("CATEGORIA")).alias("category"),
                             F.col("UBIGEO").alias("ubigeo"),
                             F.col("DEPARTAMENTO").alias("department"),
                             F.col("PROVINCIA").alias("province"),
                             F.col("DISTRITO").alias("district"),
                             F.col("NORTE").alias("latitude"),
                             F.col("ESTE").alias("longitude"))
                     .withColumn("care_level", F.regexp_extract("category", r"^(I+)", 1)))

health_facilities = write_table(health_facilities, "health_facilities")
print(f"{health_facilities.count():,} active facilities written")
health_facilities.groupBy("care_level", "category").count().orderBy("care_level", "category").show()

26,972 active facilities written
+----------+--------+-----+
|care_level|category|count|
+----------+--------+-----+
|      NULL|    NULL| 4914|
|         I|     I-1| 9903|
|         I|     I-2| 6781|
|         I|     I-3| 4275|
|         I|     I-4|  417|
|        II|    II-1|  230|
|        II|    II-2|   87|
|        II|    II-E|  305|
|       III|   III-1|   35|
|       III|   III-2|   15|
|       III|   III-E|   10|
+----------+--------+-----+



## 5. Weather stations (SENAMHI)

The API responses are nested JSON: each month file holds a list of days, and each day a list of
`{column, value}` pairs. Two `explode` calls flatten them to one row per station, day and variable,
and `pivot` turns the four variables back into columns. Passing the pivot values explicitly saves Spark
a job to discover them. Values arrive as numbers, nulls or `"T"` (trace rain, under 0.1 mm), so they are
read as text and `"T"` becomes 0. The station code comes from the folder name, and each station's
`ubigeo` (department + province + district codes) from its `station.json`.

In [9]:
stations = (spark.read.json(str(RAW / "senamhi" / "*" / "station.json"), multiLine=True)
            .select(F.col("data.stationId").alias("station_code"),
                    F.col("data.name").alias("station_name"),
                    F.concat("data.departmentId", "data.provinceId", "data.districtId").alias("ubigeo"),
                    F.col("data.latitude").cast("double").alias("latitude"),
                    F.col("data.longitude").cast("double").alias("longitude"),
                    F.col("data.altitude").cast("int").alias("altitude_m")))
stations.show(truncate=False)

+------------+----------------+------+--------+---------+----------+
|station_code|station_name    |ubigeo|latitude|longitude|altitude_m|
+------------+----------------+------+--------+---------+----------+
|100109      |PUERTO MALDONADO|170101|-12.5873|-69.2088 |209       |
|105100      |MIRAFLORES      |200104|-5.17528|-80.61654|34        |
|103038      |PUERTO PIZARRO  |240101|-3.50396|-80.39538|6         |
|108021      |PUCALLPA        |250101|-8.41323|-74.57185|162       |
|103057      |AMAZONAS        |160101|-3.76397|-73.25492|113       |
|100132      |SHANUSI         |160201|-6.06703|-76.2505 |154       |
|100138      |TARAPOTO        |220901|-6.47597|-76.37039|382       |
|106014      |MOYOBAMBA       |220101|-6.04481|-76.96822|879       |
|104079      |MALLARES        |200605|-4.85574|-80.73556|44        |
+------------+----------------+------+--------+---------+----------+



In [10]:
readings_raw = spark.read.json(
    str(RAW / "senamhi" / "*" / "20*.json"),
    multiLine=True,
    schema="data STRUCT<readings: ARRAY<STRUCT<date: STRING, values: ARRAY<STRUCT<column: STRING, value: STRING>>>>>")

weather_long = (readings_raw
                .withColumn("station_code", F.regexp_extract(F.col("_metadata.file_path"), r"senamhi/(\d+)/", 1))
                .select("station_code", F.explode("data.readings").alias("reading"))
                .select("station_code", F.to_date("reading.date").alias("date"), F.explode("reading.values").alias("v"))
                .select("station_code", "date", F.col("v.column").alias("variable"),
                        F.when(F.col("v.value") == "T", 0.0).otherwise(F.col("v.value").cast("double")).alias("value")))

variables = {"TEMPERATURA_MAXIMA": "tmax_c", "TEMPERATURA_MINIMA": "tmin_c",
             "HUMEDAD_RELATIVA": "rh_pct", "PRECIPITACION": "precip_mm"}
epi_year, epi_week = epi_year_week(F.col("date"))

weather = (weather_long
           .groupBy("station_code", "date")
           .pivot("variable", list(variables))
           .agg(F.first("value"))
           .withColumnsRenamed(variables)
           .join(F.broadcast(stations), "station_code")
           .withColumn("epi_year", epi_year)
           .withColumn("epi_week", epi_week))

weather_daily = write_table(weather, "weather_daily")
print(f"{weather_daily.count():,} station-days written")
(weather_daily.groupBy("station_name", "ubigeo")
 .agg(F.count("*").alias("days"),
      F.round(F.avg(F.col("precip_mm").isNotNull().cast("int")) * 100, 1).alias("pct_days_with_rain_data"),
      F.round(F.sum("precip_mm") / 4, 0).alias("avg_yearly_rain_mm"),
      F.round(F.avg("tmax_c"), 1).alias("avg_tmax_c"))
 .orderBy(F.desc("avg_yearly_rain_mm")).show(truncate=False))

12,827 station-days written
+----------------+------+----+-----------------------+------------------+----------+
|station_name    |ubigeo|days|pct_days_with_rain_data|avg_yearly_rain_mm|avg_tmax_c|
+----------------+------+----+-----------------------+------------------+----------+
|AMAZONAS        |160101|1400|94.6                   |2606.0            |32.6      |
|SHANUSI         |160201|1461|100.0                  |2579.0            |33.5      |
|PUERTO MALDONADO|170101|1461|99.9                   |2149.0            |31.5      |
|PUCALLPA        |250101|1430|100.0                  |1594.0            |32.0      |
|TARAPOTO        |220901|1461|99.2                   |1402.0            |32.1      |
|MOYOBAMBA       |220101|1461|100.0                  |1349.0            |29.2      |
|PUERTO PIZARRO  |240101|1442|91.1                   |467.0             |30.5      |
|MIRAFLORES      |200104|1310|99.9                   |195.0             |31.4      |
|MALLARES        |200605|1401|92.4   

In [11]:
# Consistency check: every date falls inside the epidemiological week computed for it.
outside = weather_daily.filter(~F.col("date").between(epi_week_start(F.col("epi_year"), F.col("epi_week")),
                                                      F.date_add(epi_week_start(F.col("epi_year"), F.col("epi_week")), 6)))
assert outside.count() == 0
weather_daily.select(F.min("date"), F.max("date"), F.min("epi_week"), F.max("epi_week")).show()

+----------+----------+-------------+-------------+
| min(date)| max(date)|min(epi_week)|max(epi_week)|
+----------+----------+-------------+-------------+
|2022-01-01|2025-12-31|            1|           53|
+----------+----------+-------------+-------------+



## Summary

Tables written to `data/processed/`, ready for the analysis notebook.

In [12]:
for name in ["dengue_cases", "population", "water_emergencies", "health_facilities", "weather_daily"]:
    table = spark.read.parquet(str(PROCESSED / name))
    print(f"{name:18} {table.count():>10,} rows | {', '.join(table.columns)}")

spark.stop()

dengue_cases        1,065,274 rows | epi_week, ubigeo, department, province, district, icd10, diagnosis_type, sex, age_years, severity, is_confirmed, age_group, week_start, year
population             75,636 rows | year, ubigeo, department, province, district, sex, age_group, population
water_emergencies       9,538 rows | ubigeo, department, province, district, year, decree, start_date, end_date, duration_days, district_population, water_coverage_pct, water_hours_per_day, description, emergency_type, is_extension


health_facilities      26,972 rows | facility_code, name, institution, classification, facility_type, category, ubigeo, department, province, district, latitude, longitude, care_level
weather_daily          12,827 rows | station_code, date, tmax_c, tmin_c, rh_pct, precip_mm, station_name, ubigeo, latitude, longitude, altitude_m, epi_year, epi_week
